### LOAD LIBRARY

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

### CLASS SVQR

In [2]:
class AccurateSVQR(BaseEstimator, RegressorMixin):
    """
    Accurate Support Vector Quantile Regression with Pinball Loss
    
    This implementation uses the correct pinball loss function and 
    a custom solver based on quadratic programming formulation.
    """
    
    def __init__(self, tau=0.5, kernel='rbf', C=1.0, gamma='auto', 
                 epsilon=1e-6, max_iter=1000, tol=1e-6):
        """
        Parameters:
        -----------
        tau : float, default=0.5
            Quantile level (0 < tau < 1)
        kernel : str, default='rbf'
            Kernel type ('linear', 'rbf', 'poly')
        C : float, default=1.0
            Regularization parameter
        gamma : float or 'auto', default='auto'
            Kernel coefficient for RBF kernel
        epsilon : float, default=1e-6
            Precision tolerance
        max_iter : int, default=1000
            Maximum iterations for solver
        tol : float, default=1e-6
            Convergence tolerance
        """
        self.tau = tau
        self.kernel = kernel
        self.C = C
        self.gamma = gamma
        self.epsilon = epsilon
        self.max_iter = max_iter
        self.tol = tol
        
    def _compute_kernel_matrix(self, X1, X2=None):
        """Compute kernel matrix"""
        if X2 is None:
            X2 = X1
            
        if self.kernel == 'linear':
            return np.dot(X1, X2.T)
        elif self.kernel == 'rbf':
            if self.gamma == 'auto':
                gamma = 1.0 / X1.shape[1]
            else:
                gamma = self.gamma
                
            # Efficient RBF kernel computation
            X1_norm = np.sum(X1**2, axis=1).reshape(-1, 1)
            X2_norm = np.sum(X2**2, axis=1).reshape(1, -1)
            distances = X1_norm + X2_norm - 2 * np.dot(X1, X2.T)
            return np.exp(-gamma * np.maximum(distances, 0))
        elif self.kernel == 'poly':
            degree = getattr(self, 'degree', 3)
            return (np.dot(X1, X2.T) + 1) ** degree
        else:
            raise ValueError(f"Unsupported kernel: {self.kernel}")
    
    def _pinball_loss(self, y_true, y_pred):
        """Pinball loss function (quantile loss)"""
        error = y_true - y_pred
        return np.mean(np.maximum(self.tau * error, (self.tau - 1) * error))
    
    def _pinball_loss_derivative(self, error):
        """Derivative of pinball loss"""
        return np.where(error >= 0, -self.tau, self.tau - 1)
    
    def _solve_dual_problem(self, K, y):
        """
        Solve the dual problem for SVQR using quadratic programming approach
        
        The dual problem for SVQR is:
        maximize: sum(y_i * (alpha_i^+ - alpha_i^-)) - 0.5 * sum_ij (alpha_i^+ - alpha_i^-)(alpha_j^+ - alpha_j^-) K_ij
        subject to: 0 <= alpha_i^+ <= C*tau, 0 <= alpha_i^- <= C*(1-tau)
                   sum(alpha_i^+ - alpha_i^-) = 0
        """
        n = len(y)
        
        # Variables: [alpha_plus, alpha_minus]
        # Each of size n, so total 2n variables
        
        def objective(alpha):
            alpha_plus = alpha[:n]
            alpha_minus = alpha[n:]
            alpha_diff = alpha_plus - alpha_minus
            
            # Objective: -[sum(y_i * alpha_diff_i) - 0.5 * alpha_diff^T K alpha_diff]
            linear_term = np.dot(y, alpha_diff)
            quadratic_term = 0.5 * np.dot(alpha_diff, np.dot(K, alpha_diff))
            return -(linear_term - quadratic_term)
        
        def objective_grad(alpha):
            alpha_plus = alpha[:n]
            alpha_minus = alpha[n:]
            alpha_diff = alpha_plus - alpha_minus
            
            grad_diff = -y + np.dot(K, alpha_diff)
            grad = np.zeros(2*n)
            grad[:n] = grad_diff      # gradient w.r.t alpha_plus
            grad[n:] = -grad_diff     # gradient w.r.t alpha_minus
            return grad
        
        # Constraints
        constraints = []
        
        # Sum constraint: sum(alpha_plus - alpha_minus) = 0
        A_eq = np.zeros((1, 2*n))
        A_eq[0, :n] = 1    # alpha_plus coefficients
        A_eq[0, n:] = -1   # alpha_minus coefficients
        b_eq = np.array([0])
        
        constraints.append({
            'type': 'eq',
            'fun': lambda alpha: np.dot(A_eq, alpha) - b_eq,
            'jac': lambda alpha: A_eq
        })
        
        # Bounds: 0 <= alpha_plus <= C*tau, 0 <= alpha_minus <= C*(1-tau)
        bounds = []
        for i in range(n):
            bounds.append((0, self.C * self.tau))        # alpha_plus bounds
        for i in range(n):
            bounds.append((0, self.C * (1 - self.tau)))  # alpha_minus bounds
        
        # Initial guess
        alpha0 = np.zeros(2*n)
        
        # Solve optimization problem
        result = minimize(
            objective, alpha0, method='SLSQP',
            jac=objective_grad, bounds=bounds, constraints=constraints,
            options={'maxiter': self.max_iter, 'ftol': self.tol}
        )
        
        if not result.success:
            print(f"Warning: Optimization did not converge: {result.message}")
        
        alpha_plus = result.x[:n]
        alpha_minus = result.x[n:]
        self.dual_coef_ = alpha_plus - alpha_minus
        
        # Find support vectors (non-zero alphas)
        support_threshold = 1e-6
        self.support_mask_ = (np.abs(self.dual_coef_) > support_threshold)
        self.support_vectors_ = self.X_fit_[self.support_mask_]
        self.support_coef_ = self.dual_coef_[self.support_mask_]
        self.n_support_ = np.sum(self.support_mask_)
        
        return result
    
    def _compute_bias(self, K, y):
        """Compute bias term using KKT conditions"""
        # For SVQR, bias computation is more complex than standard SVR
        # We use support vectors that are not at the bounds
        
        predictions_no_bias = np.dot(K, self.dual_coef_)
        
        # Find support vectors not at bounds for bias computation
        alpha_plus = np.maximum(self.dual_coef_, 0)
        alpha_minus = np.maximum(-self.dual_coef_, 0)
        
        # Support vectors not at upper bounds
        not_at_upper_plus = (alpha_plus < self.C * self.tau - 1e-6) & (alpha_plus > 1e-6)
        not_at_upper_minus = (alpha_minus < self.C * (1 - self.tau) - 1e-6) & (alpha_minus > 1e-6)
        
        bias_candidates = []
        
        if np.any(not_at_upper_plus):
            idx = np.where(not_at_upper_plus)[0]
            for i in idx:
                bias_candidates.append(y[i] - predictions_no_bias[i])
        
        if np.any(not_at_upper_minus):
            idx = np.where(not_at_upper_minus)[0]
            for i in idx:
                bias_candidates.append(y[i] - predictions_no_bias[i])
        
        if bias_candidates:
            self.intercept_ = np.mean(bias_candidates)
        else:
            # Fallback: use median of all residuals
            residuals = y - predictions_no_bias
            self.intercept_ = np.median(residuals)
    
    def fit(self, X, y):
        """Fit the SVQR model"""
        self.X_fit_ = X.copy()
        self.y_fit_ = y.copy()
        
        print(f"Fitting SVQR with tau={self.tau}, C={self.C}, kernel={self.kernel}")
        
        # Compute kernel matrix
        K = self._compute_kernel_matrix(X)
        
        # Solve dual problem
        self._solve_dual_problem(K, y)
        
        # Compute bias
        self._compute_bias(K, y)
        
        print(f"Training completed. Support vectors: {self.n_support_}/{len(y)}")
        
        return self
    
    def predict(self, X):
        """Predict quantile values"""
        if not hasattr(self, 'dual_coef_'):
            raise ValueError("Model not fitted yet")
        
        K = self._compute_kernel_matrix(X, self.X_fit_)
        predictions = np.dot(K, self.dual_coef_) + self.intercept_
        
        return predictions
    
    def score(self, X, y):
        """Score using pinball loss (lower is better)"""
        y_pred = self.predict(X)
        return -self._pinball_loss(y, y_pred)  # Negative because higher score is better

### SVQR 0.01

In [3]:
import pandas as pd

ret = pd.read_excel("D:/HASRI/EMPIRIS/Data Empiris(ok).xlsx")
sig = pd.read_excel("D:/HASRI/EMPIRIS/variabel_signifikan_1(3).xlsx")
var = pd.read_excel("D:/HASRI/EMPIRIS/var_empiris_1(3).xlsx")

# daftar semua nama variabel (39 kolom)
all_vars = var.columns.tolist()

# siapkan dataframe kosong untuk hasil prediksi
prediksi = pd.DataFrame(index=var.index, columns=all_vars)

for j in range(len(sig)):
    
    # nama respon = nama kolom ke-j
    response_name = sig.iloc[j, 0]

    # baris signifikan
    row = sig.iloc[j]

    # ambil nama prediktor signifikan
    selected = row.index[(row == 1)].tolist()
    selected = [c for c in selected if c != "Response"]

    # X dan y
    X = var[selected].values
    y = ret[response_name].tail(422).values

    # model SVQR
    model = AccurateSVQR(tau=0.01, C=1.0, kernel='rbf', gamma='auto')
    model.fit(X, y)

    # prediksi
    y_pred = model.predict(X)

    # masukkan hasil ke kolom yang sesuai (kolom = nama respon)
    prediksi[response_name] = y_pred

# simpan ke Excel
prediksi.to_excel("D:/HASRI/EMPIRIS/prediksi_1(3).xlsx", index=False)

Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.01, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422

### SVQR 0.05

In [4]:
import pandas as pd

ret = pd.read_excel("D:/HASRI/EMPIRIS/Data Empiris(ok).xlsx")
sig = pd.read_excel("D:/HASRI/EMPIRIS/variabel_signifikan_5(3).xlsx")
var = pd.read_excel("D:/HASRI/EMPIRIS/var_empiris_5(3).xlsx")

# daftar semua nama variabel (39 kolom)
all_vars = var.columns.tolist()

# siapkan dataframe kosong untuk hasil prediksi
prediksi = pd.DataFrame(index=var.index, columns=all_vars)

for j in range(len(sig)):
    
    # nama respon = nama kolom ke-j
    response_name = sig.iloc[j, 0]

    # baris signifikan
    row = sig.iloc[j]

    # ambil nama prediktor signifikan
    selected = row.index[(row == 1)].tolist()
    selected = [c for c in selected if c != "Response"]

    # X dan y
    X = var[selected].values
    y = ret[response_name].tail(422).values

    # model SVQR
    model = AccurateSVQR(tau=0.05, C=1.0, kernel='rbf', gamma='auto')
    model.fit(X, y)

    # prediksi
    y_pred = model.predict(X)

    # masukkan hasil ke kolom yang sesuai (kolom = nama respon)
    prediksi[response_name] = y_pred

# simpan ke Excel
prediksi.to_excel("D:/HASRI/EMPIRIS/prediksi_5(3).xlsx", index=False)

Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.05, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422

### SVQR 0.10

In [5]:
import pandas as pd

ret = pd.read_excel("D:/HASRI/EMPIRIS/Data Empiris(ok).xlsx")
sig = pd.read_excel("D:/HASRI/EMPIRIS/variabel_signifikan_10(3).xlsx")
var = pd.read_excel("D:/HASRI/EMPIRIS/var_empiris_10(3).xlsx")

# daftar semua nama variabel (39 kolom)
all_vars = var.columns.tolist()

# siapkan dataframe kosong untuk hasil prediksi
prediksi = pd.DataFrame(index=var.index, columns=all_vars)

for j in range(len(sig)):
    
    # nama respon = nama kolom ke-j
    response_name = sig.iloc[j, 0]

    # baris signifikan
    row = sig.iloc[j]

    # ambil nama prediktor signifikan
    selected = row.index[(row == 1)].tolist()
    selected = [c for c in selected if c != "Response"]

    # X dan y
    X = var[selected].values
    y = ret[response_name].tail(422).values

    # model SVQR
    model = AccurateSVQR(tau=0.10, C=1.0, kernel='rbf', gamma='auto')
    model.fit(X, y)

    # prediksi
    y_pred = model.predict(X)

    # masukkan hasil ke kolom yang sesuai (kolom = nama respon)
    prediksi[response_name] = y_pred

# simpan ke Excel
prediksi.to_excel("D:/HASRI/EMPIRIS/prediksi_10(3).xlsx", index=False)

Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SVQR with tau=0.1, C=1.0, kernel=rbf
Training completed. Support vectors: 422/422
Fitting SV